# Recreating GEMbLeR: promoter and terminator shuffling with Cre

Cautereels et al. (2024), *Combinatorial optimization of gene expression through recombinase-mediated promoter and terminator shuffling in yeast*, Nature Communications 15, 1112.

A gene is flanked by two arrays of regulatory blocks. The 5' array holds six upstream promoter elements (UPEs) in front of a fixed TDH3 core promoter, and the 3' array holds six terminators behind the coding sequence. Every block is separated by a 34 bp LoxPsym site, and the sites of one array share a specificity. Inducing Cre makes each cell cut some pair of sites and join the ends again, so the array is left with some blocks deleted, some turned around and some in a new order. A population of cells then holds a library of expression levels, which the authors screened for fluorescence (the yECitrine reporter) and for astaxanthin production (six pathway genes at six loci).

`SequenceGraph.recombine` writes a Cre event as one strand exchange between two sites that already exist, so the whole library of products can live in one graph beside the parent. Called on its own it is one cell, and each molecule it leaves becomes a graph of its own. Stacked, it keeps the parent beside every product, which is how the library is built. The notebook follows the paper:

| Part | Paper | What is built here |
|------|-------|--------------------|
| 1 | Figs 1 and 2: choosing parts | alternatives at one position of a construct |
| 2 | Fig 3a, d: 5' GEM (UPE shuffling) | one array in front of yECitrine |
| 3 | Fig 3b, e: 3' GEM (terminator shuffling) | one array behind yECitrine |
| 4 | Fig 3c, f: both arrays | two arrays with different site specificities in one graph |
| 5 | Fig 3d to f: Sanger-sequenced clones | the 107 deposited reads checked against the graphs |
| 6 | Fig 4: astaxanthin, ySTART | six genes, twelve arrays, twelve specificities |

The sequences are the published ones (Supplementary Data 1). `gembler_parts.fa` holds the parts used here, `gembler_sanger_reads.fa` the deposited Sanger reads (ENA PRJNA1000506) and `gembler_pathway_events.tsv` the counts behind Fig 4i.

## Setup and parts

LoxPsym is a loxP site whose 8 bp spacer is palindromic. A loxP site has a direction, so two sites in one molecule either excise or invert the segment between them, depending on how they face. A LoxPsym site reads the same on both strands, so it has no direction, and any two sites of one specificity can do either. That is why a GEM array produces deletions and inversions from the same sites. Different arrays use sites with different three-base spacer variants, which the paper designs so that they do not cross-react.

In [1]:
import itertools
import math
import tempfile
from collections import Counter
from csv import DictReader
from pathlib import Path

import gen

repository = gen.Repository(tempfile.mkdtemp(prefix="gembler-"))

COMPLEMENT = str.maketrans("ACGT", "TGCA")


def reverse_complement(sequence):
    return sequence.translate(COMPLEMENT)[::-1]


def read_fasta(path, label_field=0):
    records, label = {}, None
    for line in Path(path).read_text().splitlines():
        if line.startswith(">"):
            label = line[1:].split()[label_field]
            records[label] = ""
        else:
            records[label] += line.strip().upper()
    return records


PARTS = read_fasta("gembler_parts.fa")
SITES = {name.removeprefix("lox_"): sequence for name, sequence in PARTS.items() if name.startswith("lox_")}

for name, sequence in SITES.items():
    arm, spacer = sequence[:13], sequence[13:21]
    print(f"LoxPsym-{name}  {arm} {spacer} {sequence[21:]}  palindromic: {sequence == reverse_complement(sequence)}")

LoxPsym-GTC  ATAACTTCGTATA GTCTAGAC TATACGAAGTTAT  palindromic: True
LoxPsym-GGG  ATAACTTCGTATA GGGTACCC TATACGAAGTTAT  palindromic: True
LoxPsym-TTA  ATAACTTCGTATA TTATATAA TATACGAAGTTAT  palindromic: True
LoxPsym-TAG  ATAACTTCGTATA TAGTACTA TATACGAAGTTAT  palindromic: True
LoxPsym-AGA  ATAACTTCGTATA AGATATCT TATACGAAGTTAT  palindromic: True
LoxPsym-TAA  ATAACTTCGTATA TAATATTA TATACGAAGTTAT  palindromic: True
LoxPsym-GGC  ATAACTTCGTATA GGCTAGCC TATACGAAGTTAT  palindromic: True
LoxPsym-TCA  ATAACTTCGTATA TCATATGA TATACGAAGTTAT  palindromic: True
LoxPsym-GCC  ATAACTTCGTATA GCCTAGGC TATACGAAGTTAT  palindromic: True
LoxPsym-TTG  ATAACTTCGTATA TTGTACAA TATACGAAGTTAT  palindromic: True
LoxPsym-CAC  ATAACTTCGTATA CACTAGTG TATACGAAGTTAT  palindromic: True
LoxPsym-GGT  ATAACTTCGTATA GGTTAACC TATACGAAGTTAT  palindromic: True


### Building and marking a cassette

`import_library` assembles named parts in one operation. Here every column has one part: a fixed
flank or an array block with the two half-sites around it. This leaves each Cre cut at a part
boundary. We then search the assembled parent path for each full LoxPsym motif and persist the
hits as annotations. The cut positions below come from those annotations, so they remain tied
to graph bases after recombination.
Labels show only the specificity: `L-TTA` for a lone site, or `L-TTA'1`, `L-TTA'2`, and so on when it occurs more than once in a graph. The one-quote suffix is unique across arrays of the same specificity.


In [2]:
class Array:
    """Blocks separated by LoxPsym sites of one specificity."""

    def __init__(self, specificity, blocks):
        self.specificity = specificity
        self.site = SITES[specificity]
        self.blocks = blocks
        self.annotations = []


def build_cassette(name, sample, parts, part_prefix=None):
    """Import fixed columns, then annotate the recognition sites found by graph search."""
    pieces, current, arrays = [], "", []
    for part in parts:
        if isinstance(part, str):
            current += part
            continue
        left_half, right_half = part.site[:17], part.site[17:]
        pieces.append(current + left_half)
        first_piece = len(pieces) - 1
        for block in part.blocks.values():
            pieces.append(right_half + block + left_half)
        current = right_half
        arrays.append((part, first_piece))
    pieces.append(current)

    prefix = part_prefix if part_prefix is not None else f"{name}_part_"
    columns = [[gen.Sequence(f"{prefix}{index}", piece)] for index, piece in enumerate(pieces)]
    sequence_graph = repository.import_library(name, columns, sample=sample)
    label_totals = Counter()
    for array, _ in arrays:
        label_totals[array.specificity] += len(array.blocks) + 1
    label_counts = Counter()
    for array, first_piece in arrays:
        matches = sequence_graph.search(array.site, sequence_kind="exact")
        for index in range(len(array.blocks) + 1):
            site_start = sum(map(len, pieces[:first_piece + index + 1])) - 17
            expected = sequence_graph.locus(site_start, site_start + len(array.site))
            matching = [match for match in matches if match.start() == expected.start() and match.end() == expected.end()]
            assert len(matching) == 1, (name, array.specificity, index, len(matching))
            label_counts[array.specificity] += 1
            label = f"L-{array.specificity}"
            if label_totals[array.specificity] > 1:
                label += f"'{label_counts[array.specificity]}"
            array.annotations.append(sequence_graph.add_annotation(
                matching[0], label, track="LoxPsym"
            ))
    return sequence_graph


### One Cre event, and all of them

A `Site` is a stretch of sequence an enzyme acts on and the place each strand is cut inside it. A LoxPsym site is cut between its two half-sites, which is 17 bases in, on both strands. `recombine(site_a, site_b)` exchanges the strands across two sites. Two sites in one molecule that read the same way excise the segment between them as a circle. Writing the second one as `site.reverse_complement()` aligns the same two sites the other way round, and the segment is turned around instead.

The library needs no list of events. Cre acts on every site it can find, and a recombination can create sites that did not exist before: where two half-sites meet after a deletion, the junction spells a full LoxPsym site again, and searching for the site sequence finds it. `cre_closure` searches the graph, recombines every pair of sites both ways, and searches again until no new pair is left. Every event is stacked, because a population always holds cells that have not reacted, so the parent stays beside every product.

In [3]:
CUT = 17  # a LoxPsym site is cut between its half-sites, 17 bases in


def site_key(site):
    """Name a physical site by its bases, the same from either strand."""
    forward = tuple(repr(site.locus[index]) for index in range(len(site.locus)))
    mirror = site.reverse_complement()
    return min(forward, tuple(repr(mirror.locus[index]) for index in range(len(mirror.locus))))


def cre_closure(sequence_graph, site_sequence):
    """Let Cre act on every pair of sites of one specificity, until a search finds no new pair."""
    done = set()
    while True:
        hits = sequence_graph.search(site_sequence, sequence_kind="exact")
        sites = [gen.Site(hit, CUT) for hit in hits]
        keys = [site_key(site) for site in sites]
        pending = []
        for (first, first_key), (second, second_key) in itertools.combinations(zip(sites, keys), 2):
            # A junction found on both strands is one site, so a pair is its two physical sites.
            pair = frozenset((first_key, second_key))
            if first_key != second_key and pair not in done:
                done.add(pair)
                pending.append((first, second))
        if not pending:
            return len(done)
        for first, second in pending:
            sequence_graph.recombine(first, second, stack=True)
            sequence_graph.recombine(first, second.reverse_complement(), stack=True)

### Reading an array back as blocks

An enumerated path is described by the ordered, oriented blocks that survive, written `TDH3 -PGI1` for TDH3 forward then PGI1 reversed. `layout` reads that off a sequence. `cre_reachable` independently tracks deletion of any run or reversal of any run, repeated as often as needed. It covers only layouts with at most one copy of each starting block, so the comparisons below do not assess duplications or translocations.

In [4]:
def flip(label):
    return label[1:] if label.startswith("-") else "-" + label


def layout(sequence, array):
    labels = {}
    for name, block in array.blocks.items():
        labels[block] = name
        labels[reverse_complement(block)] = "-" + name
    position = sequence.find(array.site)
    found = []
    while sequence.startswith(array.site, position):
        position += len(array.site)
        for block, label in labels.items():
            if sequence.startswith(block, position):
                found.append(label)
                position += len(block)
                break
        else:
            break
    return tuple(found)


def cre_reachable(names):
    start = tuple(names)
    seen, frontier = {start}, [start]
    while frontier:
        following = []
        for state in frontier:
            for i in range(len(state) + 1):
                for j in range(i + 1, len(state) + 1):
                    deleted = state[:i] + state[j:]
                    inverted = state[:i] + tuple(flip(label) for label in reversed(state[i:j])) + state[j:]
                    for product in (deleted, inverted):
                        if product not in seen:
                            seen.add(product)
                            following.append(product)
        frontier = following
    return seen

## 1. Choosing parts

The first experiments compare parts in a fixed construct, one at a time. `import_library`
expresses each screen as named alternatives in one column; fixed columns hold the shared
sequence. Fig 2a screens 13 UPEs, Fig 2d screens 11 terminators, and Supplementary Fig 2b
crosses three UPEs with three core promoters.


In [5]:
site = SITES["TTA"]
left_half, right_half = site[:17], site[17:]
yecitrine, cyc1 = PARTS["yECitrine"], PARTS["terminator_CYC1_full"]

upe_names = [name.removeprefix("screen_upe_") for name in PARTS if name.startswith("screen_upe_")]
upe_screen = repository.import_library("upe_screen", [
    [gen.Sequence("left_site", left_half)],
    [gen.Sequence(name, right_half + PARTS[f"screen_upe_{name}"] + left_half) for name in upe_names],
    [gen.Sequence("reporter", right_half + PARTS["core_TDH3"] + yecitrine + cyc1)],
], sample="part_screens")
print("UPE screen:", len(upe_screen.all_sequences()), "constructs for", len(upe_names), "UPEs")

terminator_names = [name.removeprefix("screen_terminator_") for name in PARTS if name.startswith("screen_terminator_")]
terminator_screen = repository.import_library("terminator_screen", [
    [gen.Sequence("promoter_reporter", PARTS["promoter_TPI1_full"] + yecitrine + left_half)],
    [gen.Sequence(name, right_half + PARTS[f"screen_terminator_{name}"] + left_half) for name in terminator_names],
    [gen.Sequence("right_site", right_half)],
], sample="part_screens")
print("terminator screen:", len(terminator_screen.all_sequences()), "constructs for", len(terminator_names), "terminators")

architecture = repository.import_library("architecture", [
    [gen.Sequence("left_site", left_half)],
    [gen.Sequence("upe_" + name, right_half + PARTS[f"screen_upe_{name}"] + left_half)
     for name in ("TDH3", "PGI1", "TPI1")],
    [gen.Sequence("core_" + name, right_half + PARTS[f"core_{name}"])
     for name in ("TDH3", "PGI1", "TPI1")],
    [gen.Sequence("reporter", yecitrine + cyc1)],
], sample="part_screens")
print("promoter architecture:", len(architecture.all_sequences()), "constructs (3 UPEs x 3 cores)")


UPE screen: 13 constructs for 13 UPEs
terminator screen: 11 constructs for 11 terminators
promoter architecture: 9 constructs (3 UPEs x 3 cores)


## 2. Promoter shuffling: the 5' GEM (Fig 3a, d)

The reporter strain carries the six-UPE array in front of the TDH3 core promoter, yECitrine and the CYC1 terminator at the CAN1 locus. The first six UPEs are TDH3, PGI1, TPI1, CCW12, ALD6 and RPL13B (the last with its 12 bp linker). The genomic flanks are left out.

A small version first, with three blocks, to see what the stack of events looks like:

In [ ]:
UPES = {name: PARTS[f"upe_{name}"] for name in ["TDH3", "PGI1", "TPI1", "CCW12", "ALD6", "RPL13B"]}
REPORTER_TAIL = PARTS["core_TDH3"] + PARTS["yECitrine"] + PARTS["terminator_CYC1_full"]

preview_array = Array("TTA", dict(list(UPES.items())[:3]))
preview = build_cassette("promoter_preview", "promoter_shuffle", [preview_array, REPORTER_TAIL], part_prefix="p")
print("marked LoxPsym sites:", [annotation.name for annotation in preview_array.annotations])
cre_closure(preview, preview_array.site)
preview.plot(interactive=False, rows=200, cols=900, detail="labels")

In [ ]:
preview.plot(interactive=False, rows=200, cols=900, detail="minimal")

### One cell

Stacking keeps every possible product in one graph. A single cell does not: it has one molecule, and a Cre event leaves it with new molecules. Without `stack`, `recombine` is final, the original junctions are deleted, and each molecule it leaves is a graph of its own.

Excising the middle block of a three-block array gives the cassette without it and a circle that holds the block. The circle is named after the cassette with a number, and it starts at the recombination site, so its coordinates count from the site.

In [ ]:
cell_array = Array("TTA", dict(list(UPES.items())[:3]))
cell = build_cassette("one_cell", "single_cell", [cell_array, REPORTER_TAIL], part_prefix="c")
cell_sites = [gen.Site(annotation.locus, CUT) for annotation in cell_array.annotations]

excised = cell.recombine(cell_sites[1], cell_sites[2], split=True)
print("graphs after the excision:", [graph.name for graph in excised])
print("cassette layout:", layout(str(excised[0].all_sequences()[0]), cell_array))
circle = str(excised[1].all_sequences()[0])
print("circle holds PGI1:", cell_array.blocks["PGI1"] in circle, "| circle length:", len(circle), "bp")
print("graphs of the sample:", [graph.name for graph in repository.sample("single_cell")])

Reversing the second site lines the same two sites up the other way round. The segment between them is turned around instead of cut out, so the molecule stays in one graph.

In [ ]:
inverted_cell_array = Array("TTA", dict(list(UPES.items())[:3]))
inverted_cell = build_cassette("one_cell_inverted", "single_cell_inverted", [inverted_cell_array, REPORTER_TAIL], part_prefix="i")
inverted_sites = [gen.Site(annotation.locus, CUT) for annotation in inverted_cell_array.annotations]

inverted = inverted_cell.recombine(
    inverted_sites[1], inverted_sites[2].reverse_complement(), split=True
)
print("graphs after the inversion:", [graph.name for graph in inverted])
print("cassette layout:", layout(str(inverted[0].all_sequences()[0]), inverted_cell_array))

The excision edge that closes a segment into a circle also makes a cycle in this stacked graph. A walk that keeps the parent junction, loops through that segment and then exits reads a tandem repeat. It can loop again for more copies. This is a graph walk, not proof that one intramolecular Cre event produced that linear molecule: integration of the excised circle into a copy of the segment would be needed for copy gain. `all_sequences()` enumerates simple paths and therefore omits walks that revisit the cycle. The next cell checks this on the three-block preview.

In [ ]:
block = preview_array.blocks["PGI1"]
direct_repeat = block + preview_array.site + block
inverted_repeat = block + preview_array.site + reverse_complement(block)
assert preview.search(direct_repeat, sequence_kind="exact")
assert preview.search(inverted_repeat, sequence_kind="exact")
for repeat in (direct_repeat, inverted_repeat):
    assert all(repeat not in str(sequence) for sequence in preview.all_sequences())
print("forward tandem-repeat walk: present; omitted by all_sequences()")
print("inverted tandem-repeat walk: present; omitted by all_sequences()")

A tandem inverted duplication (`PGI1 -PGI1`, or the opposite order) is biologically possible if a circular copy integrates in the opposite orientation next to a retained copy; LoxPsym allows either orientation. The closure includes it. The excised circle is a loop in this graph, its sites can be found like any other, and recombining one of them with a site of the retained copy is that integration, aligned either way round. So the stacked graph holds the junction for both orientations, and `all_sequences()` still lists only the simple paths. Now the real array with six blocks: `all_sequences()` lists its simple-path products.

In [ ]:
promoter_array = Array("TTA", UPES)
promoter_shuffle = build_cassette("reporter_5prime", "promoter_shuffle_full", [promoter_array, REPORTER_TAIL])
parent_sequence = str(promoter_shuffle.all_sequences()[0])
cre_closure(promoter_shuffle, promoter_array.site)

promoter_layouts = [layout(str(sequence), promoter_array) for sequence in promoter_shuffle.all_sequences()]
reachable = cre_reachable(UPES)
print("sequences in the graph:", len(promoter_layouts), "distinct layouts:", len(set(promoter_layouts)))
print("layouts Cre can make from six blocks:", len(reachable))
print("graph layouts that Cre cannot make:", len(set(promoter_layouts) - reachable))
print("blocks kept:", sorted(Counter(len(state) for state in promoter_layouts).items()))
print("no UPE left:", promoter_layouts.count(()), "| parent:", layout(parent_sequence, promoter_array))

Every sequence returned by `all_sequences()` is a deletion/inversion layout in the independent bookkeeping, and every layout that bookkeeping reaches is in the graph, so the closure finds the whole library: deleting any run of blocks, turning around any run, and repeating either as often as needed. The reported count is for enumerated simple paths, not for every walk through the graph. In particular, the excision cycles permit tandem-repeat walks that this count excludes. The paper reports few inversions: 5.6% of the recombination events in the reporter clones (Fig 3g).

Three examples from the enumerated library, each checked against its own sequence:

In [ ]:
def describe(array, layouts, wanted):
    for state in wanted:
        print(f"{' '.join(state) or '(no UPE)':<28}", "in graph:", state in layouts)


describe(promoter_array, set(promoter_layouts), [
    ("TDH3", "PGI1", "TPI1", "CCW12", "ALD6", "RPL13B"),
    ("TDH3", "-TPI1", "ALD6"),
    ("-RPL13B",),
    (),
])

## 3. Terminator shuffling: the 3' GEM (Fig 3b, e)

Same construction on the other side of the gene: the TPI1 promoter, yECitrine, then six terminators (ACS2, ADH1, VSP13, CYC1, IDP1, HIS5) separated by sites of a second specificity, TAG.

In [ ]:
TERMINATORS = {name: PARTS[f"terminator_{name}"] for name in ["ACS2", "ADH1", "VSP13", "CYC1", "IDP1", "HIS5"]}

terminator_array = Array("TAG", TERMINATORS)
terminator_shuffle = build_cassette(
    "reporter_3prime", "terminator_shuffle", [PARTS["promoter_TPI1_full"] + PARTS["yECitrine"], terminator_array]
)
cre_closure(terminator_shuffle, terminator_array.site)

terminator_layouts = [layout(str(sequence), terminator_array) for sequence in terminator_shuffle.all_sequences()]
print("sequences in the graph:", len(terminator_layouts), "| layouts Cre cannot make:", len(set(terminator_layouts) - cre_reachable(TERMINATORS)))

## 4. Both arrays together (Fig 3c, f)

In the combined strain the same gene carries the 5' array (specificity TTA) and the 3' array (TAG). The two specificities do not cross-react, so events act inside one array and never between them. Each array is then an independent choice, and the library is the product of the two.

The check is a small cassette with two blocks per array, small enough to list every sequence:

In [ ]:
small_upes = Array("TTA", dict(list(UPES.items())[:2]))
small_terminators = Array("TAG", dict(list(TERMINATORS.items())[:2]))
small = build_cassette(
    "combined_small", "combined", [small_upes, PARTS["core_TDH3"] + PARTS["yECitrine"], small_terminators]
)
cre_closure(small, small_upes.site)
cre_closure(small, small_terminators.site)

pairs = {(layout(str(s), small_upes), layout(str(s), small_terminators)) for s in small.all_sequences()}
upe_states = {upe for upe, _ in pairs}
terminator_states = {terminator for _, terminator in pairs}
print(len(small.all_sequences()), "sequences =", len(upe_states), "x", len(terminator_states), "->", len(pairs) == len(upe_states) * len(terminator_states))
small.plot(interactive=True, rows=60, cols=900, show_history=False, detail="labels")

Orthogonality is what lets the arrays be independent. If both arrays carried the same specificity, which is what the paper's generic standalone arrays do before the pathway design gives each array its own, a site in the 5' array could pair with one in the 3' array and delete the gene between them. Adding those events to the same small cassette shows it:

In [ ]:
shared_five = Array("TTA", dict(list(UPES.items())[:2]))
shared_three = Array("TTA", dict(list(TERMINATORS.items())[:2]))
shared = build_cassette("shared_sites", "combined", [shared_five, PARTS["core_TDH3"] + PARTS["yECitrine"], shared_three])
# One specificity, so the search finds the sites of both arrays and the closure pairs them across.
cre_closure(shared, shared_five.site)

with_gene = sum(PARTS["yECitrine"] in str(sequence) for sequence in shared.all_sequences())
print(len(shared.all_sequences()), "sequences,", len(shared.all_sequences()) - with_gene, "of them have lost yECitrine")
print("with orthogonal sites,", sum(PARTS["yECitrine"] not in str(s) for s in small.all_sequences()), "of", len(small.all_sequences()), "have lost it")

For the full reporter the library is the 5' layouts times the 3' layouts. The experiment itself sampled it, with 180 clones screened and 23 sequenced:

In [ ]:
combined_array_5 = Array("TTA", UPES)
combined_array_3 = Array("TAG", TERMINATORS)
combined = build_cassette("reporter_both", "combined_full", [combined_array_5, PARTS["core_TDH3"] + PARTS["yECitrine"], combined_array_3])
cre_closure(combined, combined_array_5.site)
cre_closure(combined, combined_array_3.site)
print(f"{len(set(promoter_layouts))} x {len(set(terminator_layouts))} = {len(set(promoter_layouts)) * len(set(terminator_layouts)):,} layouts in one graph that was never enumerated")

## 5. The sequenced clones (Fig 3d to f)

The authors Sanger-sequenced 23 clones per design and drew their layouts. The deposited reads are single reads across the array from either end. Here each read is aligned to the graphs with minigraph, so the graph itself reads the layout: the path of an alignment through the segments says which junctions the clone took.

`gembler_sanger_reads.gaf` holds the alignments of all 107 reads, made with `minigraph -cx lr` against the GFA of the 5' and 3' reporter graphs built above, with `gembler_sanger_reads.fa` as the reads. Segment names in a GAF are content hashes of the sequences, so the file matches only graphs with these segments. `load_gaf` places each read on the graph it belongs to and counts the others as skipped. The reads include genomic flanks that the graphs leave out, so an alignment covers the cassette part of a read and the flanks are clipped.

In [ ]:
import re

SANGER_GAF = "gembler_sanger_reads.gaf"
READS = read_fasta("gembler_sanger_reads.fa", label_field=1)
read_names = {}
for line in Path("gembler_sanger_reads.fa").read_text().splitlines():
    if line.startswith(">"):
        accession, name, *_ = line[1:].split()
        read_names[accession] = name

reporters = {"5' reporter": promoter_shuffle, "3' reporter": terminator_shuffle}
placements = {label: graph.load_gaf(SANGER_GAF) for label, graph in reporters.items()}
for label, placed in placements.items():
    print(f"{label}: {len(placed)} reads placed | on the other reporter: {placed.skipped} | untraceable: {dict(placed.untraceable)}")

An alignment is a walk through the graph: its steps are oriented segments, and consecutive steps are the junctions the read crosses. The shape of the walk classifies the clone without calling blocks. A walk that changes strand crosses an inverted segment, and one that revisits a segment goes around a cycle, which is a tandem copy.

In [ ]:
def parse_gaf(path):
    """The alignments of each read: where in the read, and the oriented segments of the path."""
    alignments = {}
    for line in Path(path).read_text().splitlines():
        fields = line.split("\t")
        alignments.setdefault(fields[0], []).append({
            "start": int(fields[2]),
            "end": int(fields[3]),
            "steps": re.findall(r"([<>])([^<>]+)", fields[5]),
        })
    return alignments


alignments = parse_gaf(SANGER_GAF)
for label, placed in placements.items():
    shapes = Counter()
    for name in placed:
        parts = alignments[name]
        steps = max(parts, key=lambda part: part["end"] - part["start"])["steps"]
        segments = [segment for _, segment in steps]
        shapes["reads placed"] += 1
        shapes["cross a junction"] += len(steps) > 1
        shapes["change strand (inversion)"] += len({orientation for orientation, _ in steps}) > 1
        shapes["revisit a segment (tandem copy)"] += len(set(segments)) < len(segments)
    print(label)
    for shape, count in shapes.items():
        print(f"  {shape:<34} {count}")

The graph holds every junction Cre can make, and the reads show which of them occurred. Keeping only the links that a read crosses gives a graph of what was observed. It is a new graph in the repository, imported from a GFA with the unobserved links dropped, and each of its segments is annotated with the number of reads that cover it.

In [ ]:
def observed_graph(label, graph, sample):
    """A new graph with only the segments and links that a placed Sanger read crosses."""
    library_gfa = Path(tempfile.mkdtemp(prefix="gembler-observed-")) / "library.gfa"
    graph.export_gfa(str(library_gfa))
    records = [line.split("\t") for line in library_gfa.read_text().splitlines()]
    sequences = {record[1]: record[2] for record in records if record[0] == "S"}
    opposite = {"+": "-", "-": "+"}
    strand = {">": "+", "<": "-"}
    links, depth = set(), Counter()
    for name in placements[label]:
        covered = set()
        for alignment in alignments[name]:
            steps = alignment["steps"]
            for (first_strand, first), (second_strand, second) in zip(steps, steps[1:]):
                links.add((first, strand[first_strand], second, strand[second_strand]))
                links.add((second, opposite[strand[second_strand]], first, opposite[strand[first_strand]]))
            covered.update(segment for _, segment in steps)
        depth.update(covered)
    kept = [f"S\t{name}\t{sequences[name]}" for name in depth]
    kept += ["\t".join(record) for record in records if record[0] == "L" and tuple(record[1:5]) in links]
    observed_gfa = library_gfa.with_name("observed.gfa")
    observed_gfa.write_text("\n".join(kept) + "\n")
    observed = repository.import_gfa(str(observed_gfa), sample=sample)
    for name, reads in depth.items():
        hits = observed.search(sequences[name], sequence_kind="exact")
        if len(hits) == 0:
            break
        observed.add_annotation(hits[0], f"{reads} reads, {name[:6]}", track="Sanger reads")
    return observed


observed_graphs = {}
for label, sample in (("5' reporter", "observed_5prime"), ("3' reporter", "observed_3prime")):
    observed_graphs[label] = observed_graph(label, reporters[label], sample)
    library_edges = reporters[label].to_networkx().number_of_edges()
    print(f"{label}: {observed_graphs[label].to_networkx().number_of_edges()} of the library's {library_edges} edges were observed")
observed_graphs["5' reporter"].plot(interactive=False, rows=60, cols=900, detail="minimal")

### Reads that map in pieces

A read that the graph explains in one piece maps once. A read that carries a junction the graph lacks, or sequence it does not hold, maps in pieces, with a stretch between them that aligns to nothing. Counting the reads with split mapping is the check for mystery sequences.

In [ ]:
split_reads = {accession: parts for accession, parts in alignments.items() if len(parts) > 1}
unaligned = [name for accession, name in read_names.items() if accession not in alignments]
print(f"reads with an alignment: {len(alignments)} of {len(read_names)} | split mapping: {len(split_reads)} | no alignment: {len(unaligned)}")
for accession, parts in split_reads.items():
    ordered = sorted(parts, key=lambda part: part["start"])
    gaps = [later["start"] - earlier["end"] for earlier, later in zip(ordered, ordered[1:])]
    print(f"  {read_names[accession]}: pieces at {[(part['start'], part['end']) for part in ordered]}, {gaps} unaligned bases between")

### A real read crosses a recombinant junction

The alignments summarize many reads, but a direct graph search gives stronger evidence.
This deposited Sanger read contains an exact 124 bp window crossing a LoxPsym site between
inverted blocks. The window is absent from either orientation of the parent. The read is
sequenced opposite to the graph's stored orientation, so we search its reverse complement
with `sequence_kind="exact"` and check the returned locus bases. The full read can contain
Sanger errors; this clean window tests the junction directly.


In [ ]:
read_name = "SangerSeq_GEMvariant_1.D6-22.0_file_b"
read = READS[read_name]
site_start = read.index(SITES["TTA"])
read_window = read[site_start - 45:site_start + len(SITES["TTA"]) + 45]
assert len(read_window) == 124
assert read_window not in parent_sequence
assert read_window not in reverse_complement(parent_sequence)
graph_query = reverse_complement(read_window)
read_hits = promoter_shuffle.search(graph_query, sequence_kind="exact")
assert len(read_hits) == 1 and read_hits[0].sequence == graph_query
print(read_name)
print(len(read_window), "read bases across LoxPsym; exact graph hits:", len(read_hits))


Reads that show the same block at two positions are not simple paths of the library, but the aligner places them anyway: their alignments revisit a segment, which is a walk around one of the graph's cycles. The paper counts translocations and duplications separately from deletions and inversions.

## 6. Six genes, twelve arrays (Fig 4)

The astaxanthin strain ySTART has six genes at six loci, each with its own 5' and 3' array. The genes are tHMG1, CrtE, CrtI, CrtYB, CrtW and CrtZ. The twelve arrays use twelve different LoxPsym specificities, so every array recombines on its own. The coding sequences and the TDH3 core promoter stay fixed. Each locus is one graph, the six together are one sample, and the left flank of each is the 30 to 50 bp integration tail that the paper's oligonucleotide gives (it maps to the genomic site named in the paper).

The unrecombined cassettes can be checked against the supplement. The published lengths are 4811, 4364, 4982, 5255, 3959 and 3722 bp, excluding the tails:

In [10]:
SPECIFICITIES = {
    "tHMG1": ("GTC", "GGG"),
    "CrtE": ("TTA", "TAG"),
    "CrtI": ("AGA", "TAA"),
    "CrtYB": ("GGC", "TCA"),
    "CrtW": ("GCC", "TTG"),
    "CrtZ": ("CAC", "GGT"),
}
PUBLISHED_LENGTH = {"tHMG1": 4811, "CrtE": 4364, "CrtI": 4982, "CrtYB": 5255, "CrtW": 3959, "CrtZ": 3722}

pathway = {}
for gene, (five_prime, three_prime) in SPECIFICITIES.items():
    arrays = (Array(five_prime, UPES), Array(three_prime, TERMINATORS))
    tail = PARTS[f"tail_{gene}"]
    sequence_graph = build_cassette(gene, "ySTART", [tail, arrays[0], PARTS["core_TDH3"] + PARTS[f"cds_{gene}"], arrays[1]])
    cassette_length = len(str(sequence_graph.all_sequences()[0])) - len(tail)
    assert cassette_length == PUBLISHED_LENGTH[gene], (gene, cassette_length)
    pathway[gene] = (sequence_graph, arrays)
    print(f"{gene:<6} {five_prime} / {three_prime}  {cassette_length} bp, matches the supplement")

assert len({array.site for _, arrays in pathway.values() for array in arrays}) == 12

tHMG1  GTC / GGG  4811 bp, matches the supplement
CrtE   TTA / TAG  4364 bp, matches the supplement
CrtI   AGA / TAA  4982 bp, matches the supplement
CrtYB  GGC / TCA  5255 bp, matches the supplement
CrtW   GCC / TTG  3959 bp, matches the supplement
CrtZ   CAC / GGT  3722 bp, matches the supplement


Now every array gets its closure. Nothing is added between genes or between the two arrays of a gene, which is what the orthogonal sites guarantee:

In [ ]:
import time

started = time.time()
for gene, (sequence_graph, arrays) in pathway.items():
    for array in arrays:
        cre_closure(sequence_graph, array.site)
    print(f"{gene:<6} both arrays done, {time.time() - started:,.0f} s so far")

states_per_array = len(set(promoter_layouts))
print("sample graphs:", [graph.name for graph in repository.sample("ySTART")])
print(f"layouts per array: {states_per_array:,} (5') and {len(set(terminator_layouts)):,} (3')")
print(f"layouts per gene: {states_per_array * len(set(terminator_layouts)):,}")
print(f"genotypes of the whole strain: 10^{12 * math.log10(states_per_array):.1f}")

The experiment screened 1,408 Cre-treated colonies and sequenced 30, so it sampled
this space very thinly. The sequencing counted which events happened in each array (Fig 4i).
Those counts measure how often Cre acted; the graph records which routes are possible.


In [12]:
events = Counter()
for row in DictReader(open("gembler_pathway_events.tsv"), delimiter="\t"):
    events[row["Target gene"], row["construct"], row["Recombination event"]] += int(row["value"])

for kind in ("deletion", "inversion", "translocation", "duplication"):
    print(f"{kind:<14} total {sum(v for (_, _, e), v in events.items() if e == kind)}")
print()
print("deletions   5' array   3' array")
for gene in ("tHMG", "CrtE", "CrtI", "CrtYB", "CrtW", "CrtZ"):
    print(f"{gene:<10} {events[gene, '5GEM', 'deletion']:>8} {events[gene, '3GEM', 'deletion']:>10}")

deletion       total 347
inversion      total 84
translocation  total 7
duplication    total 4

deletions   5' array   3' array
tHMG            134          6
CrtE             39          7
CrtI             31         20
CrtYB            36          5
CrtW             40          9
CrtZ              4         16


In [ ]:
# export_gfa writes the entire sample, even when called on one sequence graph.
roundtrip_gfa = Path(tempfile.mkdtemp(prefix="gembler-gfa-")) / "ystart.gfa"
sequence_graph.export_gfa(str(roundtrip_gfa))
with roundtrip_gfa.open() as handle:
    gfa_lines = handle.read().splitlines()
gfa_counts = Counter(line[0] for line in gfa_lines if line)
print("GFA records:", dict(gfa_counts))
print("original sample graphs:", [graph.name for graph in repository.sample("ySTART")])

# Re-exporting unchanged data should preserve each link's strand fields.
repeat_gfa = roundtrip_gfa.with_name("ystart_repeat.gfa")
sequence_graph.export_gfa(str(repeat_gfa))
links = {line for line in gfa_lines if line.startswith("L\t")}
repeat_links = {line for line in repeat_gfa.read_text().splitlines() if line.startswith("L\t")}
print("links changed by a repeated export:", len(links.symmetric_difference(repeat_links)))
assert links == repeat_links, "Repeated export should preserve oriented links"


In [ ]:
roundtrip_graph = repository.import_gfa(str(roundtrip_gfa), sample="ySTART_gfa")
print("imported sample graphs:", [graph.name for graph in repository.sample("ySTART_gfa")])
print("GFA named paths:", [line.split("\t")[1] for line in roundtrip_gfa.read_text().splitlines() if line.startswith("P\t")])


In [ ]:
# The GFA importer merges six loci into one graph with one pair of terminal nodes.
original_networks = [graph.to_networkx() for graph in repository.sample("ySTART")]
imported_network = roundtrip_graph.to_networkx()
original_nodes = sum(graph.number_of_nodes() for graph in original_networks)
original_edges = sum(graph.number_of_edges() for graph in original_networks)
print("six original graphs: ", original_nodes, "nodes /", original_edges, "directed node pairs")
print("one imported graph: ", imported_network.number_of_nodes(), "nodes /", imported_network.number_of_edges(), "directed node pairs")
assert imported_network.number_of_nodes() == original_nodes - 2 * (len(original_networks) - 1)


def physical_connections(networks):
    """Compare strand-aware junctions independently of node IDs and mirror spellings."""
    sequences = [node.sequence for network in networks for node in network if node.sequence]
    # Sequence labels identify nodes in these cassettes; fail rather than merge repeated segments.
    assert len(sequences) == len(set(sequences)), "Connection comparison requires unique segment sequences"
    complement = {"+": "-", "-": "+"}
    connections = set()
    for network in networks:
        for source, target, attributes in network.edges(data=True):
            # Import assigns new segment IDs but keeps the two shared terminal IDs.
            source_key = ("sequence", source.sequence) if source.sequence else ("terminal", str(source._id))
            target_key = ("sequence", target.sequence) if target.sequence else ("terminal", str(target._id))
            for edge in attributes["attr_dict"]:
                source_strand, target_strand = edge["source_strand"], edge["target_strand"]
                forward = (source_key, target_key, source_strand, target_strand)
                mirror = (target_key, source_key, complement[target_strand], complement[source_strand])
                connections.add(min(forward, mirror))
    return connections


# Export collapses a+ -> b+ and b- -> a- into one physical connection.
# Include all strand pairs and self-links, so losing a hairpin still fails this check.
original_connections = physical_connections(original_networks)
imported_connections = physical_connections([imported_network])
assert original_connections == imported_connections, (
    f"Missing {len(original_connections - imported_connections)} physical connections; "
    f"added {len(imported_connections - original_connections)}"
)
print("physical connections preserved:", len(original_connections))

# This read crosses a recombinant junction: run CrtE's closure above before exporting.
original_pathway_hits = pathway["CrtE"][0].search(graph_query, sequence_kind="exact")
roundtrip_hits = roundtrip_graph.search(graph_query, sequence_kind="exact")
assert len(original_pathway_hits) == 1 and original_pathway_hits[0].sequence == graph_query
assert len(roundtrip_hits) == 1 and roundtrip_hits[0].sequence == graph_query
print("recombinant read window: original", len(original_pathway_hits), "/ imported", len(roundtrip_hits))


The recombinant read window requires CrtE’s 5′ closure. Run the pathway closure
cell before exporting, then rerun import and the checks.

Export includes all six graphs in the sample; import merges them into one graph with
one shared pair of terminal nodes. Export also collapses reverse-complement spellings
of the same physical connection, so raw directed edge counts can differ. The roundtrip
check compares every strand-aware connection modulo that symmetry, including hairpins,
and verifies the recombinant read window in both graphs. Segment sequences are unique
in these cassettes, allowing comparison despite the new node IDs assigned by import.


## 7. Following one nanopore read through the graph

Minigraph aligns a read to the exported GFA and writes one GAF line per alignment. `load_gaf` turns each read in that file into a `Locus`, the same object a search returns, so a read can be shown, sliced and compared like any other region. `gembler_reads.gaf` holds three reads from the deposited nanopore run SRR25462427 (ENA PRJNA1000506), aligned with `minigraph -cx lr` to the GFA exported above. Segment names in a GAF are content hashes of the sequences, so the file matches only this graph.

Reads that do not lie in this graph are counted in `skipped`. A read that takes a step the graph has no edge for is listed in `untraceable` and is not loaded.

In [ ]:
thmg1 = pathway["tHMG1"][0]
reads = thmg1.load_gaf("gembler_reads.gaf")
print("loaded:", len(reads), "| skipped:", reads.skipped, "| unmapped:", reads.unmapped, "| untraceable:", dict(reads.untraceable))
for name in reads:
    print(f"{name:<16} {len(reads[name]):>5} bases  strand {reads[name].strand}")


A read keeps the strand it travelled. A read that crosses an inverted segment reports `"mixed"`, because part of its walk runs on the reverse strand. Showing a read lights only the edges it crosses. Parallel edges between the same two segments, such as the same junction read on the other strand, stay dim, so the picture shows which recombination link the read used.

In [ ]:
import IPython.display

read_view = thmg1.plot(detail="minimal", rows=40, cols=400)
#read_view.show(reads["SRR25462427.2"], color="red")
read_view.show(reads["SRR25462427.14"], color="cyan")
read_png = Path(tempfile.mkdtemp(prefix="gembler-read-")) / "read.png"
read_view.save_png(read_png, rows=60, cols=400)
IPython.display.Image(filename=str(read_png))


In [ ]:
import IPython.display

read_view = thmg1.plot(detail="minimal", rows=40, cols=400)
read_view.show(reads["SRR25462427.2"], color="red")
#read_view.show(reads["SRR25462427.14"], color="cyan")
read_png = Path(tempfile.mkdtemp(prefix="gembler-read-")) / "read.png"
read_view.save_png(read_png, rows=100, cols=600)
IPython.display.Image(filename=str(read_png))


### A path that changes strand

The read above crosses one inverted segment. To see the edge choice on its own, take the three-block preview from section 2 and a molecule that keeps TDH3, inverts PGI1 and keeps TPI1. Its path leaves TDH3 on the forward strand, runs through PGI1 on the reverse strand and comes back to the forward strand. Each hop uses one particular edge, the one that leaves the previous block's strand and enters the next block's. The graph holds other edges between the same blocks, such as the one that skips PGI1, and those stay dim.

In [ ]:
inverted_middle = next(
    str(sequence)
    for sequence in preview.all_sequences()
    if layout(str(sequence), preview_array) == ("TDH3", "-PGI1", "TPI1")
)
site_starts = [at for at in range(len(inverted_middle)) if inverted_middle.startswith(preview_array.site, at)]
across_both_junctions = inverted_middle[site_starts[1] - 60 : site_starts[2] + len(preview_array.site) + 60]
path_hits = preview.search(across_both_junctions, sequence_kind="exact")
assert len(path_hits) == 1 and path_hits[0].strand == "mixed"

path_view = preview.plot(detail="labels", rows=60, cols=900)
path_view.show(path_hits[0], color="#FF5F1F")
path_png = Path(tempfile.mkdtemp(prefix="gembler-path-")) / "path.png"
path_view.save_png(path_png, rows=60, cols=900)
IPython.display.Image(filename=str(path_png))